In [0]:
import pandas as pd
df = pd.read_csv("flight_data_100k.csv")
df

In [0]:
df.dtypes
df1 = df.copy()
df1['FL_DATE'] = pd.to_datetime(df1['FL_DATE'])
df1.dtypes

In [0]:
df1

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

df1['month'] = df1['FL_DATE'].dt.month

monthXairline = (df1.groupby(["month", "AIRLINE"])["CANCELLED"].mean().unstack()*100)

plt.figure(figsize=(12, 8))
sns.heatmap(monthXairline, cmap="YlGnBu", annot=True, fmt=".1f",linewidths=0.5)
plt.show()

In [0]:

originXairline = (df1.groupby(["ORIGIN", "AIRLINE"])["CANCELLED"].mean().unstack()*100)

plt.figure(figsize=(12, 100))
sns.heatmap(originXairline, cmap="YlGnBu", annot=True, fmt=".1f",linewidths=0.5)
plt.show()

In [0]:
df1['day'] = df1['FL_DATE'].dt.day

dayXairline = (df1.groupby(["day", "AIRLINE"])["CANCELLED"].mean().unstack()*100)

plt.figure(figsize=(12, 100))
sns.heatmap(dayXairline, cmap="YlGnBu", annot=True, fmt=".1f",linewidths=0.5)
plt.show()

In [0]:

destXairline = (df1.groupby(["DEST", "AIRLINE"])["CANCELLED"].mean().unstack()*100)

plt.figure(figsize=(12, 100))
sns.heatmap(destXairline, cmap="YlGnBu", annot=True, fmt=".1f",linewidths=0.5)
plt.show()

In [0]:


hourXairline = (df1.groupby(["DEP_TIME", "AIRLINE"])["CANCELLED"].mean().unstack()*100)

plt.figure(figsize=(12, 100))
sns.heatmap(hourXairline, cmap="YlGnBu", annot=True, fmt=".1f",linewidths=0.5)
plt.show()

In [0]:
df1.isnull().sum()

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,average_precision_score
from sklearn.model_selection import train_test_split


df1 = df1.sort_values(by = 'FL_DATE')
split_80 = df1.iloc[:int(len(df1) * 0.8)]

df1["monthXairline"] = (
    df1.groupby(["month", "AIRLINE"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)

       .fillna(split_80["CANCELLED"].mean() * 100)
)

df1["destXairline"] = (
    df1.groupby(["DEST", "AIRLINE"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)

       .fillna(split_80["CANCELLED"].mean() * 100)
)


df1['dayofweek'] = df1['FL_DATE'].dt.dayofweek

df1["dayofweek_cancel"] = (
    df1.groupby(["dayofweek"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)

       .fillna(split_80["CANCELLED"].mean() * 100)
)

df1["originXcancel"] = (
    df1.groupby(["ORIGIN"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)
       
       .fillna(split_80["CANCELLED"].mean() * 100)
)

df1["originxdestination"] = (
    df1.groupby(["ORIGIN", "DEST"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)
       
       .fillna(split_80["CANCELLED"].mean() * 100)
)

df1['hour'] = (df1['CRS_DEP_TIME'] // 100)
df1["hourXairline"] = (
    df1.groupby(["hour", "AIRLINE"])["CANCELLED"]
       .transform(lambda x: x.shift(1).expanding().mean() * 100)
       
       .fillna(split_80["CANCELLED"].mean() * 100)
)


df1["quarter"] = df1["FL_DATE"].dt.quarter
df1["quarterXairline"] = (
    df1.groupby(["quarter", "AIRLINE"])["CANCELLED"]
    .transform(lambda x: x.shift(1).expanding().mean() * 100)

    .fillna(split_80["CANCELLED"].mean() * 100)
)


day_cancel = df1.groupby("FL_DATE")["CANCELLED"].mean().reset_index()

day_cancel["1day_cancellationrate"] = (
    day_cancel["CANCELLED"].shift(1)
)

day_cancel["1day_cancellationrate"] = (
    day_cancel["1day_cancellationrate"]
    .fillna(split_80["CANCELLED"].mean())
)
df1 = df1.merge(
    day_cancel[["FL_DATE", "1day_cancellationrate"]], on="FL_DATE", how="left"
)

df1["datetime"] = (
    df1["FL_DATE"] +
    pd.to_timedelta(df1["CRS_DEP_TIME"] // 100, unit="h")
)


hour_cancel = df1.groupby("datetime")["CANCELLED"].mean()

df1["previous_hour"] = df1["datetime"] - pd.Timedelta(hours=1)

df1["eachhour_cancellationrate"] = (
    df1["previous_hour"]
    .map(hour_cancel)
    .fillna(split_80["CANCELLED"].mean())
)


features = ['monthXairline','destXairline','dayofweek_cancel','originXcancel','originxdestination','DISTANCE','hourXairline', 'quarterXairline','1day_cancellationrate','eachhour_cancellationrate']
target = 'CANCELLED'


X_train, X_test, y_train, y_test = train_test_split(df1[features], df1[target], test_size=0.2, shuffle = False)





In [0]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train1 = scaler.fit_transform(X_train)
X_test1 = scaler.transform(X_test)

model = LogisticRegression(class_weight="balanced")
model.fit(X_train1, y_train)
y_prediction = model.predict(X_test1)
y_probability = model.predict_proba(X_test1)[:, 1]


print(classification_report(y_test, y_prediction))
print(confusion_matrix(y_test, y_prediction))
print(roc_auc_score(y_test, y_probability))


In [0]:
%pip install lightgbm

In [0]:
import lightgbm as lgb

model = lgb.LGBMClassifier(class_weight="balanced",random_state=42)

model.fit(X_train, y_train)

y_prediction = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_prediction))
print(confusion_matrix(y_test, y_prediction))
print(roc_auc_score(y_test, y_probability))
print(average_precision_score(y_test, y_probability))

In [0]:
df1

